# Zero-Shot SBIR: Kaggle GPU Training & Cross-Dataset Evaluation Pipeline
**Architecture:** Asymmetric Hybrid LoRA-CLIP (ViT-B/32)
**Training Dataset:** Sketchy Database (Split 2: 104 Seen Classes)
**Testing Datasets:**
1. Sketchy Unseen (21 Classes) - In-Domain Zero-Shot Benchmark
2. TU-Berlin Extended (30 Unseen Classes or 250 Classes) - Cross-Dataset Generalization Benchmark

## Step 1: Clone Repository & Install Dependencies
Select **GPU T4 x2** or **P100** in Kaggle Notebook settings (Settings -> Accelerator -> GPU).

In [ ]:
!git clone https://github.com/Quandinh030106/ZS-SBIR-LoRA-CLIP.git
%cd ZS-SBIR-LoRA-CLIP
!pip install -q -r requirements.txt
!python train.py --self-test

## Step 2: Discover Attached Datasets in Kaggle
Inspect the paths to your attached Sketchy and TU-Berlin datasets under `/kaggle/input/`.

In [ ]:
import os
print("Available input datasets in /kaggle/input/:")
for root, dirs, files in os.walk("/kaggle/input/"):
    depth = root.replace("/kaggle/input/", "").count(os.sep)
    if depth <= 2:
        print("  " * depth + f"- {os.path.basename(root)}/")

## Step 3: Evaluate Baseline 0 (Frozen CLIP ViT-B/32 on Sketchy Split 2)
Measures pure zero-shot performance before training to establish the experimental reference line.

In [ ]:
# Update path below to match your attached Sketchy dataset path
SKETCHY_ROOT = "/kaggle/input/sketchy-dataset/256x256"

!python train.py \
  --data-root "$SKETCHY_ROOT" \
  --split split2 \
  --eval-baseline-only

## Step 4: Pre-compute Offline Photo Features (Cache)
Pre-computing 512D embeddings for all 10,400 seen photos eliminates disk I/O bottlenecks and accelerates training by 4-6x.

In [ ]:
CACHE_PATH = "/kaggle/working/photo_features_cache.pt"

!python scripts/cache_photo_features.py \
  --data-root "$SKETCHY_ROOT" \
  --split split2 \
  --output "$CACHE_PATH"

## Step 5: Train Asymmetric LoRA-CLIP Model (15 Epochs)
Trains with In-Batch Semi-Hard Mining, Partial Sketch Simulation (CVPR 2020), Noise-Tolerance (CVPR 2022), and Photo Caching.

In [ ]:
!python train.py \
  --data-root "$SKETCHY_ROOT" \
  --split split2 \
  --epochs 15 \
  --batch-size 64 \
  --lr 2e-5 \
  --lora-rank 8 \
  --mining-mode semi_hard \
  --lambda-cls 0.3 \
  --lambda-consist 1.0 \
  --use-cached-photos

## Step 6: Cross-Dataset Generalization Test on TU-Berlin
Evaluates the best checkpoint (trained strictly on Sketchy) directly on TU-Berlin Extended to prove domain generalization without overfitting.

In [ ]:
# Update path below to your attached TU-Berlin dataset
TUBERLIN_ROOT = "/kaggle/input/tu-berlin-dataset"
BEST_CKPT = "outputs/checkpoints/best_seco_lora_clip.pt"

if os.path.exists(TUBERLIN_ROOT):
    !python scripts/evaluate.py \
      --checkpoint "$BEST_CKPT" \
      --data-root "$TUBERLIN_ROOT" \
      --dataset tuberlin \
      --split tuberlin \
      --viz-samples 10 \
      --output-dir "/kaggle/working/tuberlin_eval_results"
else:
    print("[NOTE] TU-Berlin path not found. Attach TU-Berlin dataset to Kaggle notebook to run cross-eval.")

## Step 7: Checkpoint Verification & Ready for Web Demo
Download `best_seco_lora_clip.pt` from the Kaggle Output file browser on the right panel.

In [ ]:
import os
ckpt = "outputs/checkpoints/best_seco_lora_clip.pt"
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / (1024 * 1024)
    print(f"SUCCESS: Checkpoint ready ({size_mb:.2f} MB) at: {ckpt}")
    print("Download this file to your local computer to power the Gradio Web Demo!")
else:
    print("Checkpoint not yet generated.")